# Solar filament segmentation: end-to-end pipeline

This notebook runs the whole pipeline of our MAGFiLO 2026 submission, from the data to a
checked `submission.csv`:

1. set up and check the environment and data,
2. (optional) train the final model,
3. score the model on our held-out validation split with the organizers' PQ,
4. predict on the test images and write `submission.csv`,
5. check the file for the problems that make Kaggle reject it,
6. look at example predictions.

**Final model:** torchvision Mask R-CNN (ResNet50-FPN, COCO-pretrained), flip augmentation,
1280 px input, 16 epochs, score threshold 0.8, minimum mask area 200 px, overlaps removed.

**Before you run it:** install the packages from `requirements.txt`, and download the competition data so that `MAGFiLO_1.0_Kaggle_2026/train` and `.../test` exist next to this notebook (the data are not included in the repository). The final model weights are stored in this repository (`weights/maskrcnn_1280_final_fp16.pth`, saved in fp16 to stay under GitHub's 100 MB file limit and cast to fp32 when loaded), so no training is needed. Run time: about 5 minutes on a GPU.

In [ ]:
import sys
import subprocess
from pathlib import Path

import numpy as np
import torch
import torchvision
import matplotlib.pyplot as plt

DATA_ROOT = Path("MAGFiLO_1.0_Kaggle_2026")      # folder that contains train/ and test/
CKPT = Path("weights/maskrcnn_1280_final_fp16.pth")   # final model weights, stored in this repository
IMG_SIZE, SCORE_THRESH, MIN_AREA = 1280, 0.8, 200

RUN_TRAINING = False   # True: retrain from scratch (about 2 hours on a 6 GB GPU)
QUICK = False          # True: validate on 10 images only, for a fast smoke test

print("torch", torch.__version__, "| torchvision", torchvision.__version__,
      "| cuda available:", torch.cuda.is_available())
assert (DATA_ROOT / "train").exists() and (DATA_ROOT / "test").exists(), \
    "Data not found: put the dataset next to this notebook or edit DATA_ROOT."


def run(*args):
    """Run one of our scripts and stream its output into the notebook."""
    proc = subprocess.Popen([sys.executable, "-u", *map(str, args)],
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        print(line, end="")
    if proc.wait() != 0:
        raise RuntimeError(f"{args[0]} failed (see output above)")

## 1. Look at the data

Each training sample is one annotator's pass over one image (several annotators may have
outlined the same image; we keep every pass, as the organizers instruct). A filament is one
polygon, rasterized here to a binary mask.

In [ ]:
from dataset import FilamentDataset

train_dir = DATA_ROOT / "train"
ds = FilamentDataset(train_dir / "train_images",
                     train_dir / "MAGFiLO_1.0_Annotations_kaggle2026_train.json")
n_test = len(list((DATA_ROOT / "test" / "test_images").iterdir()))
print(f"{len(ds)} annotated training samples, {n_test} test images")

sample = ds[0]
image = sample["image"][0].numpy()
masks = sample["masks"].numpy().astype(bool)
fig, ax = plt.subplots(1, 2, figsize=(10, 5))
ax[0].imshow(image, cmap="gray"); ax[0].set_title("image"); ax[0].axis("off")
ax[1].imshow(image, cmap="gray")
ax[1].imshow(np.ma.masked_where(~masks.any(0), masks.any(0)), cmap="autumn", alpha=0.8)
ax[1].set_title(f"{len(masks)} annotated filaments"); ax[1].axis("off")
plt.show()

## 2. Train (optional)

Leave `RUN_TRAINING = False` to use the weights stored in the repository (no training needed). Set it to `True` to retrain
with the final recipe: SGD with warmup and cosine decay, batch size 1, mixed precision,
random flips, 16 epochs at 1280 px. The validation split is 10% of the *files* (seed 0), so
no image appears in both training and validation.

In [ ]:
if RUN_TRAINING:
    run("train.py", "--root", DATA_ROOT, "--epochs", 16, "--img-size", IMG_SIZE,
        "--batch-size", 1, "--augment", "--out", "checkpoints_retrain")
    CKPT = Path("checkpoints_retrain/epoch_16.pth")      # use the freshly trained weights
else:
    print("Skipping training; using the weights stored in the repository:", CKPT)

assert CKPT.exists(), f"Checkpoint not found: {CKPT}"

## 3. Score on the validation split

`validate.py` reproduces the organizers' PQ (pooled TP/FP/FN over images, a match is
IoU > 0.5) after making predicted masks disjoint, and sweeps the score threshold and
minimum area. Our final settings (score >= 0.8, area >= 200) come from this table.

In [ ]:
args = ["validate.py", "--root", DATA_ROOT, "--ckpt", CKPT, "--img-size", IMG_SIZE]
if QUICK:
    args += ["--limit", 10]
run(*args)

## 4. Predict on the test images

One CSV row per predicted filament: `filament_id` (`<image>_<k>`) and `segmentation_rle`
(compressed RLE counts of the 2048 x 2048 mask). Masks within an image are made disjoint
because the evaluation rejects overlaps. The script also prints the inference time.

In [ ]:
run("inference.py", "--root", DATA_ROOT, "--ckpt", CKPT, "--img-size", IMG_SIZE,
    "--score-thresh", SCORE_THRESH, "--min-area", MIN_AREA, "--out", "submission.csv")

## 5. Check the submission file

Verifies the header, unique ids, that every RLE decodes at the right size, that ids match
test image names, and that **no two masks in an image overlap** (it must say 0).

In [ ]:
run("check_submission.py", "--root", DATA_ROOT, "--csv", "submission.csv")

### Compare with the submitted predictions

`submission_1280.csv` (in this repository) is the file we uploaded to Kaggle (public PQ 0.36). The
predictions regenerated above should match it: the same filament ids and near-identical masks. Tiny
pixel differences can appear because the stored weights are rounded to fp16.

In [ ]:
import csv
from pycocotools import mask as mask_util


def load_rows(path):
    with open(path) as f:
        reader = csv.reader(f)
        next(reader)
        return {fid: rle for fid, rle in reader}


new, ref = load_rows("submission.csv"), load_rows("submission_1280.csv")
common = sorted(new.keys() & ref.keys())
ious = []
for fid in common:
    a = {"size": [2048, 2048], "counts": new[fid].encode()}
    b = {"size": [2048, 2048], "counts": ref[fid].encode()}
    ious.append(float(mask_util.iou([a], [b], [0])[0][0]))

print(f"regenerated: {len(new)} filaments | submitted: {len(ref)} filaments | ids in both: {len(common)}")
if ious:
    print(f"mean IoU between regenerated and submitted masks with the same id: {np.mean(ious):.4f}")

## 6. Example predictions

Ground truth (left) against our prediction (right) on one validation image.

In [ ]:
from model import build_model, predict
from train import split_by_filename

device = "cuda" if torch.cuda.is_available() else "cpu"
model = build_model(pretrained=False, img_size=IMG_SIZE)
model.load_state_dict(torch.load(CKPT, map_location="cpu"))
model.to(device).eval()
model.roi_heads.score_thresh = SCORE_THRESH      # only expand confident masks to full size

_, val_idx = split_by_filename(ds, 0.1, 0)
sample = ds[val_idx[0]]
x = sample["image"].repeat(3, 1, 1).to(device)
res = predict(model, [x], score_thresh=SCORE_THRESH)[0]
pred = res["masks"].numpy()
pred = pred[pred.sum((1, 2)) >= MIN_AREA]
gt = sample["masks"].numpy().astype(bool)
gt = gt[gt.any((1, 2))]


def overlay(ax, img, masks, title):
    ax.imshow(img, cmap="gray")
    colors = plt.get_cmap("tab10").colors
    for i, m in enumerate(masks):
        rgba = np.zeros(m.shape + (4,))
        rgba[m] = (*colors[i % 10], 0.7)
        ax.imshow(rgba)
    ax.set_title(title); ax.axis("off")


fig, ax = plt.subplots(1, 2, figsize=(12, 6))
overlay(ax[0], sample["image"][0].numpy(), gt, f"annotation ({len(gt)} filaments)")
overlay(ax[1], sample["image"][0].numpy(), pred, f"prediction ({len(pred)} filaments)")
plt.tight_layout(); plt.show()

## Results recorded for this model

| | Value |
|---|---|
| Local validation PQ (score >= 0.8, area >= 200) | 0.394 |
| Public leaderboard PQ | 0.36 |
| Inference time (180 test images, laptop RTX 4050) | 52 s (0.29 s per image) |

Human-vs-human PQ on training files with several annotators (`annotator_ceiling.py`): 0.34.
See the report for the full analysis and its limitations.